In [5]:
# Import
import random
import numpy as np
import torch
import json
from tqdm import tqdm
from pathlib import Path
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
import os
import csv
from transformers import RobertaModel, RobertaTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from collections import Counter, defaultdict, deque
import re
from rank_bm25 import BM25Okapi

device = "cuda" if torch.cuda.is_available() else "cpu"

# Seed for reproductibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Default paths
ROOT = Path("../Amazon_products") # Root Amazon_products directory
TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"

TEST_CORPUS_PATH = os.path.join(TEST_DIR, "test_corpus.txt")  # product_id \t text
TRAIN_CORPUS_PATH = os.path.join(TRAIN_DIR, "train_corpus.txt")

CLASS_HIERARCHY_PATH = ROOT / "class_hierarchy.txt" 
CLASS_RELATED_PATH = ROOT / "class_related_keywords.txt" 
CLASS_PATH = ROOT / "classes.txt" 

SUBMISSION_PATH = "../Submission/submission.csv"  # output file

# Constants
NUM_CLASSES = 531  # total number of classes (0–530)

# Loading functions
def load_classic(path):
    """Load doc into {id: text} dictionary."""
    id2text = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                id, text = parts
                id2text[id] = text
    return id2text

def load_multilabel(path):
    """Load multi-label data into {id: [labels]} dictionary -> for class_hierarchy"""
    id2labels = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                pid, label = parts
                pid = int(pid)
                label = int(label)
                if pid not in id2labels:
                    id2labels[pid] = []
                id2labels[pid].append(label)
    return id2labels

def load_class_keywords(path):
    """Load class keywords into {class_name: [keywords]} dictionary."""
    class2keywords = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if ":" not in line: # accept only valid format
                continue
            classname, keywords = line.strip().split(":", 1)
            keyword_list = [kw.strip() for kw in keywords.split(",") if kw.strip()]
            class2keywords[classname] = keyword_list
    return class2keywords

# Extraction
id2text_test = load_classic(TEST_CORPUS_PATH) # id -> text test
id_list_test = list(id2text_test.keys()) # list id test
print(list(id2text_test.items())[:1])
print(len(id_list_test)) #19658

id2text_train = load_classic(TRAIN_CORPUS_PATH) # id -> text train
id_list_train = list(id2text_train.keys()) # list id train
print(list(id2text_train.items())[:1])
print(len(id_list_train)) #29487

id2class = load_classic(CLASS_PATH) # id class -> class text
print(list(id2class.items())[:1])
print(len(id2class)) #531

class2hierarchy = load_multilabel(CLASS_HIERARCHY_PATH) # id parents -> children (taxonomy)
print(list(class2hierarchy.items())[:1])
print(len(class2hierarchy)) #69

class2related = load_class_keywords(CLASS_RELATED_PATH) # id class -> related keywords
print(list(class2related.items())[:1])
print(len(class2related)) #531


[('0', "conair cs15tcs professional straight styles straightening iron woah ! sure this straightener looks like all the other crappy straightners in the world , but there 's a twist to this one ! it is my first straightner and i 've had it for about 7 months . i bought it only because i was desperate for a cheap straightener because my hair is very thick , long , wavy ! i 'm looking for a new straighner right now ... but until then this one is doing just fine . if it works for me , it will work for you !")]
19658
[('0', 'omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use . it is nice to have immediate feedback on the bloodpressure effects of my various exercises , food consumption , and relaxation or stress levels .')]
29487
[('0', 'grocery_gourmet_food')]
531
[(0, [1, 8, 208, 211, 213, 216, 229, 255, 265, 218, 271, 277, 249, 288, 313, 357])]
69
[('grocery_gourmet_food', ['snacks'

In [14]:
import json

def hierarchy_consistency(silver, hierarchy):
    """Hierarchy consistency in a hierarchy given for our silver labels"""
    ok = 0
    total = 0
    for labels in silver.values():
        L = set(labels)
        for parent, children in hierarchy.items():
            for child in children:
                if child in L:
                    total += 1
                    if parent in L:
                        ok += 1
    return ok / total if total > 0 else 0

def label_coverage(silver_labels, num_classes=531):
    """
    silver_labels : { review_id: [label1, label2, ...] }
    returns coverage_ratio, covered_classes
    """
    covered = set()

    for i, labels in silver_labels.items():
        for lbl in labels:
            if 0 <= lbl < num_classes:
                covered.add(lbl)

    coverage_ratio = len(covered) / num_classes
    return coverage_ratio, sorted(list(covered))


def load_silver_files(paths: dict):
    """Load multiple silver files: name -> {pid -> labels}"""
    silvers = {}
    for name, path in paths.items():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        silvers[name] = {int(pid): d["labels"] for pid, d in data.items()}
        print(f"Loaded {name}: {len(silvers[name])} samples")

    return silvers

def majority_vote(silvers: dict, top_k=3, min_labels=2):
    """
    Strict majority vote: label must appear in > half of models.
    Example:
        - 4 models => majority = 3
    """
    model_count = len(silvers)
    majority = model_count // 2 + 1   # strict majority rule
    
    print(f"\nUsing strict majority: need {majority}/{model_count} votes\n")

    # only keep PIDs present in ALL models
    all_pids = set.intersection(*(set(d.keys()) for d in silvers.values()))

    clean = {}

    for pid in all_pids:
        label_counts = {}

        # count label votes across all models
        for name in silvers:
            for lab in silvers[name].get(pid, []):
                label_counts[lab] = label_counts.get(lab, 0) + 1

        # keep labels with strict majority
        vote = sorted([lab for lab, cnt in label_counts.items() if cnt >= majority])

        if len(vote) < min_labels:
            continue

        clean[pid] = {"labels": vote[:top_k]}

    print(f"Cleaned samples: {len(clean)}")
    return clean

paths = { # classic
    "mpnet": "Silver/silver_train_mpnet.json",
    "mini": "Silver/silver_train_mini.json",
}

paths2 = { # BM25
    "mpnet": "Silver/silver_train_mpnet25.json",
    "para": "Silver/silver_train_mini25.json",
}

paths3 = { # all classic + BM25
    "mpnet25": "Silver/silver_train_mpnet25.json",
    "para25": "Silver/silver_train_mini25.json",
    "mpnet": "Silver/silver_train_mpnet.json",
    "mini": "Silver/silver_train_mini.json",
}

paths4 = { # all mpnet
    "mpnet25": "Silver/silver_train_mpnet25.json",
    "mpnet": "Silver/silver_train_mpnet.json",
}

paths5 = { # all no hier more precised
    "mpnet25": "Silver/silver_train_mpnet25_nohier.json",
    "para25": "Silver/silver_train_mini25_nohier.json",
    "mpnet": "Silver/silver_train_mpnet_nohier.json",
    "mini": "Silver/silver_train_mini_nohier.json",
}

paths6 = { # mpnet nohier more precised 
    "mpnet25": "Silver/silver_train_mpnet25_nohier.json",
    "mpnet": "Silver/silver_train_mpnet_nohier.json",
}

paths7 = { # all mixed hier
    "mpnet25": "Silver/silver_train_multiMini.json",
    "mpnet":   "Silver/silver_train_multiMpnet.json",
}

# Roberta useless
# No_hier less useful for kaggle score

silvers = load_silver_files(paths7)
#silvers = load_silver_files(paths4) etc

# Majority vote
clean = majority_vote(silvers, top_k=5, min_labels=2) # do not take too much labels (in all case it will be 2 or 3)


OUT_PATH = "SilverCombo/silver_train_mixed.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(clean, f, indent=2, ensure_ascii=False)

print(f"\nSaved: {OUT_PATH}")

silver_train_labels = {pid: info["labels"] for pid, info in clean.items()}

CLASS_HIERARCHY_PATH = ROOT / "class_hierarchy.txt"
class2hierarchy = load_multilabel(CLASS_HIERARCHY_PATH)
consistency = hierarchy_consistency(silver_train_labels, class2hierarchy)
print(f"\nHierarchy Consistency: {consistency:.2%}")

coverage, classes = label_coverage(silver_train_labels)
print(f"Coverage: {coverage:.2%}")
print(f"Covered classes: {len(classes)}/531")

Loaded mpnet25: 29487 samples
Loaded mpnet: 29487 samples

Using strict majority: need 2/2 votes

Cleaned samples: 20589

Saved: SilverCombo/silver_train_mixed.json

Hierarchy Consistency: 92.22%
Coverage: 91.90%
Covered classes: 488/531
